Célula de Código 1 — Escrever o main.py

In [ ]:
%%writefile /content/main.py
# ============================================================
# API de Análise Financeira — FastAPI
# Módulo Principal (v2 — endpoints separados + nomenclatura do edital)
# ============================================================
"""
API REST que expõe o pipeline de análise financeira, seguindo
integralmente o contrato do edital do Hackathon ONE (Alura + Oracle),
incluindo nomenclatura exata de categorias no exemplo oficial e os dois
endpoints exigidos (análise financeira completa + classificação isolada
de transações).
"""

import os
import joblib
import numpy as np
import pandas as pd
import requests
from contextlib import asynccontextmanager
from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao]


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao]


NOMES_EDITAL = {
    'Alimentacao': 'alimentacao',
    'Moradia': 'moradia',
    'Transporte': 'transporte',
    'Saude': 'saude',
    'Educacao': 'educacao',
    'Lazer': 'entretenimento',
    'Servicos': 'servicos',
}

URLS_OCI = {
    'vetorizador_tfidf.pkl': 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl',
    'modelo_categoria_producao.pkl': 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl',
    'codificador_categorias.pkl': 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl',
    'modelo_perfil_producao.pkl': 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl',
    'codificador_perfil.pkl': 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl',
}
PASTA_MODELOS = '/home/ubuntu/modelos_api'


def baixar_e_carregar_artefatos() -> dict:
    """Baixa os artefatos do OCI Object Storage e carrega em memória."""
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), 'wb') as f:
            f.write(resposta.content)

    return {
        'vetorizador_tfidf': joblib.load(os.path.join(PASTA_MODELOS, 'vetorizador_tfidf.pkl')),
        'modelo_categoria': joblib.load(os.path.join(PASTA_MODELOS, 'modelo_categoria_producao.pkl')),
        'codificador_categorias': joblib.load(os.path.join(PASTA_MODELOS, 'codificador_categorias.pkl')),
        'modelo_perfil': joblib.load(os.path.join(PASTA_MODELOS, 'modelo_perfil_producao.pkl')),
        'codificador_perfil': joblib.load(os.path.join(PASTA_MODELOS, 'codificador_perfil.pkl')),
    }


def classificar_categorias_transacoes(transacoes: list, artefatos: dict) -> list:
    """Prevê a categoria de cada transação, traduzida para a nomenclatura do edital."""
    descricoes = [t['descricao'] for t in transacoes]
    vetores = artefatos['vetorizador_tfidf'].transform(descricoes)
    categorias_cod = artefatos['modelo_categoria'].predict(vetores)
    categorias_internas = artefatos['codificador_categorias'].inverse_transform(categorias_cod)

    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({
            **t,
            'categoria': NOMES_EDITAL.get(cat_interna, cat_interna.lower()),
            '_categoria_interna': cat_interna,
        })
    return resultado


def calcular_resumo_gastos(transacoes_classificadas: list) -> dict:
    """Agrupa o valor total por categoria (nomenclatura do edital)."""
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t['categoria']] += t['valor']
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas: list) -> dict:
    """Agrupa o valor total por categoria, em nomenclatura INTERNA (para o modelo de perfil)."""
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t['_categoria_interna']] += t['valor']
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos) -> tuple:
    """Prevê o perfil financeiro do usuário."""
    mapa_poupanca = {'Baixa': 0, 'Media': 1, 'Alta': 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100

    features = pd.DataFrame([{
        'renda_mensal': renda_mensal, 'nivel_endividamento': nivel_endividamento,
        'frequencia_poupanca_cod': mapa_poupanca[frequencia_poupanca],
        'comprometimento_gastos': comprometimento_gastos, **resumo_gastos_interno,
    }])

    probabilidades = artefatos['modelo_perfil'].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos['codificador_perfil'].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil: str, resumo_gastos: dict, frequencia_poupanca: str) -> list:
    """Gera recomendações baseadas em regras de negócio."""
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == 'Em risco':
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orçamento",
            "Buscar renegociação de dívidas para reduzir o nível de endividamento",
        ]
    if perfil == 'Em observacao':
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == 'Baixa':
            recs.append("Aumentar a frequência de poupança mensal")
        return recs
    return [
        "Manter o padrão atual de organização financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada: dict, artefatos: dict) -> dict:
    """Função principal: análise financeira completa."""
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada['transacoes'], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)

    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada['renda_mensal'], dados_entrada['nivel_endividamento'],
        dados_entrada['frequencia_poupanca'], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada['frequencia_poupanca'])

    return {
        'perfil_financeiro': perfil,
        'probabilidade': probabilidade,
        'resumo_gastos': {k: v for k, v in resumo_gastos_edital.items() if v > 0},
        'recomendacoes': recomendacoes,
    }


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("🔄 Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    print("✅ Modelos carregados com sucesso")
    yield
    artefatos_globais.clear()


app = FastAPI(
    title="API de Análise Financeira — G9 Team 20",
    description="Classificação de transações e perfil financeiro para o Hackathon ONE (Alura + Oracle)",
    lifespan=lifespan,
)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    """Endpoint principal: recebe dados financeiros e devolve a análise completa."""
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    return resultado


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    """Endpoint dedicado: classifica transações sem exigir dados de perfil."""
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {
        "transacoes_classificadas": [
            {"descricao": t['descricao'], "valor": t['valor'], "categoria": t['categoria']}
            for t in transacoes_classificadas
        ]
    }

Writing /content/main.py


Célula de Código 2 — Instalar dependências e testar localmente

In [ ]:
!pip install fastapi uvicorn --quiet

import sys
sys.path.insert(0, '/content')

from fastapi.testclient import TestClient
from main import app

with TestClient(app) as cliente:
    resposta_raiz = cliente.get("/")
    print("🔍 Health check:", resposta_raiz.json())

    exemplo_edital = {
        "renda_mensal": 4500,
        "nivel_endividamento": 25,
        "frequencia_poupanca": "Media",
        "transacoes": [
            {"descricao": "Supermercado", "valor": 420},
            {"descricao": "Combustivel", "valor": 300},
            {"descricao": "Streaming", "valor": 40},
        ]
    }
    resposta = cliente.post("/analise-financeira", json=exemplo_edital)
    print("\n🔍 Teste /analise-financeira:", resposta.json())

    resposta_classificar = cliente.post("/classificar-transacoes", json={"transacoes": exemplo_edital["transacoes"]})
    print("\n🔍 Teste /classificar-transacoes:", resposta_classificar.json())

    dados_invalidos = {**exemplo_edital, "transacoes": [{"descricao": "Teste", "valor": -50}]}
    resposta_invalida = cliente.post("/analise-financeira", json=dados_invalidos)
    print("\n🔍 Teste de validação (valor negativo):", resposta_invalida.status_code)

🔄 Carregando modelos do OCI Object Storage...
✅ Modelos carregados com sucesso
🔍 Health check: {'status': 'API no ar', 'modelos_carregados': True}

🔍 Teste /analise-financeira: {'perfil_financeiro': 'Saudavel', 'probabilidade': 0.87, 'resumo_gastos': {'alimentacao': 420.0, 'transporte': 300.0, 'entretenimento': 40.0}, 'recomendacoes': ['Manter o padrão atual de organização financeira', 'Considerar investir o excedente mensal para objetivos de longo prazo']}

🔍 Teste /classificar-transacoes: {'transacoes_classificadas': [{'descricao': 'Supermercado', 'valor': 420.0, 'categoria': 'alimentacao'}, {'descricao': 'Combustivel', 'valor': 300.0, 'categoria': 'transporte'}, {'descricao': 'Streaming', 'valor': 40.0, 'categoria': 'entretenimento'}]}

🔍 Teste de validação (valor negativo): 422


Célula de Código 3 — Teste dos 3 exemplos reais na API pública

In [12]:
import requests
import json

URL_API = "http://140.238.178.157:8000/analise-financeira"


def testar_exemplo(nome_cenario: str, dados_entrada: dict) -> dict:
    resposta = requests.post(URL_API, json=dados_entrada, timeout=10)
    print(f"\n{'='*60}\n📋 {nome_cenario}\n{'='*60}")
    print(f"Status HTTP: {resposta.status_code}")
    print(f"Saída: {json.dumps(resposta.json(), ensure_ascii=False, indent=2)}")
    return resposta.json()


exemplo_1 = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
    "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}, {"descricao": "Streaming", "valor": 40}]}
exemplo_2 = {"renda_mensal": 3000, "nivel_endividamento": 62, "frequencia_poupanca": "Baixa",
    "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}, {"descricao": "Posto Ipiranga", "valor": 250}, {"descricao": "Farmacia Pague Menos", "valor": 180}]}
exemplo_3 = {"renda_mensal": 5200, "nivel_endividamento": 38, "frequencia_poupanca": "Media",
    "transacoes": [{"descricao": "Mercado Atacadao", "valor": 380}, {"descricao": "Aplicativo Uber", "valor": 150}, {"descricao": "Curso Online Alura", "valor": 90}]}

resultado_1 = testar_exemplo("Exemplo 1 — Caso do edital", exemplo_1)
resultado_2 = testar_exemplo("Exemplo 2 — Endividamento alto", exemplo_2)
resultado_3 = testar_exemplo("Exemplo 3 — Endividamento moderado", exemplo_3)

ConnectionError: HTTPConnectionPool(host='140.238.178.157', port=8000): Max retries exceeded with url: /analise-financeira (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7846594dd5b0>: Failed to establish a new connection: [Errno 113] No route to host'))

Atualização de Arquivo Main junto ao Server


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ssh-key-2026-08-01.key to ssh-key-2026-08-01 (1).key


Enviar Arquivo ao Server

In [ ]:
# ============================================================
# Deploy para Produção — Notebook 05 (v2, com autocorreção de caminho)
# Módulo: Validação + Envio do main.py + Reinício da API
# ============================================================
"""
Envia o main.py mais recente para o servidor de produção via SCP, e
reinicia a API — com uma camada extra de segurança: antes de enviar,
corrige automaticamente qualquer referência ao caminho específico do
Colab (/content/...) para o caminho correto do servidor Ubuntu
(/home/ubuntu/...), evitando o erro recorrente 'PermissionError:
/content' que ocorre quando o arquivo é regerado sem essa correção.

Pré-requisito: a chave privada precisa ter sido enviada para esta
sessão do Colab via files.upload() — se a sessão foi reiniciada desde
então, é necessário fazer o upload novamente antes de rodar esta célula.
"""

import os
import paramiko
import time


def verificar_chave_disponivel(caminho_chave: str) -> bool:
    """Confirma que a chave privada já foi carregada nesta sessão do Colab."""
    existe = os.path.exists(caminho_chave)
    if not existe:
        print(f"⚠️ Chave '{caminho_chave}' não encontrada nesta sessão.")
        print("   Rode antes: from google.colab import files; files.upload()")
    return existe


def corrigir_caminho_pasta_modelos(caminho_arquivo: str = "/content/main.py") -> bool:
    """Garante que PASTA_MODELOS aponte para o caminho do servidor, não do Colab.

    Esta correção é aplicada automaticamente, antes de todo deploy,
    porque o erro 'PermissionError: /content' já ocorreu mais de uma
    vez ao regerar o main.py sem essa verificação — colocar a correção
    aqui, na própria célula de deploy, elimina a dependência de lembrar
    de rodar uma célula de correção separada.

    Returns:
        True se alguma correção foi necessária e aplicada; False se o
        arquivo já estava correto.
    """
    with open(caminho_arquivo, "r") as f:
        conteudo = f.read()

    caminho_errado = 'PASTA_MODELOS = "/content/modelos_api"'
    caminho_certo = 'PASTA_MODELOS = "/home/ubuntu/modelos_api"'

    if caminho_errado in conteudo:
        conteudo_corrigido = conteudo.replace(caminho_errado, caminho_certo)
        with open(caminho_arquivo, "w") as f:
            f.write(conteudo_corrigido)
        print("🔧 Caminho de PASTA_MODELOS corrigido automaticamente (/content → /home/ubuntu)")
        return True

    print("✅ Caminho de PASTA_MODELOS já estava correto")
    return False


def enviar_arquivo_para_servidor(ip_servidor: str, usuario: str, caminho_chave: str,
                                   caminho_local: str, caminho_remoto: str) -> None:
    """Envia um arquivo local para o servidor via SFTP."""
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    sftp = cliente_ssh.open_sftp()
    sftp.put(caminho_local, caminho_remoto)
    sftp.close()
    cliente_ssh.close()
    print(f"✅ {caminho_local} enviado para {caminho_remoto}")


def reiniciar_api_no_servidor(ip_servidor: str, usuario: str, caminho_chave: str) -> None:
    """Mata o processo antigo e sobe a versão nova, com < /dev/null para
    evitar que o processo em segundo plano derrube a conexão SSH."""
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    comando_matar = "pkill -f uvicorn; echo 'processo antigo encerrado'"
    cliente_ssh.exec_command(comando_matar)
    time.sleep(2)

    comando_subir = (
        "cd /home/ubuntu && source venv/bin/activate && "
        "nohup uvicorn main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 < /dev/null &"
    )
    cliente_ssh.exec_command(comando_subir)
    time.sleep(1)
    cliente_ssh.close()
    print("✅ Comando de reinício enviado")


def confirmar_api_no_ar(ip_servidor: str, usuario: str, caminho_chave: str) -> None:
    """Reconecta e confirma, pelo log e pela lista de processos, que a API subiu."""
    time.sleep(4)
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    stdin, stdout, stderr = cliente_ssh.exec_command("tail -20 /home/ubuntu/api.log")
    log = stdout.read().decode()
    print("📋 Últimas linhas do log:")
    print(log)

    stdin, stdout, stderr = cliente_ssh.exec_command("ps aux | grep '[u]vicorn'")
    processos = stdout.read().decode()
    print("📋 Processos ativos:")
    print(processos if processos else "⚠️ Nenhum processo uvicorn encontrado — verifique o log acima")

    cliente_ssh.close()

    if "Modelos carregados com sucesso" in log and processos:
        print("\n🎉 Deploy confirmado com sucesso!")
    else:
        print("\n⚠️ Deploy pode não ter concluído corretamente — revisar log acima")


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

if verificar_chave_disponivel(CAMINHO_CHAVE):
    corrigir_caminho_pasta_modelos("/content/main.py")
    enviar_arquivo_para_servidor(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE, "/content/main.py", "/home/ubuntu/main.py")
    reiniciar_api_no_servidor(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
    confirmar_api_no_ar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)

🔧 Caminho de PASTA_MODELOS corrigido automaticamente (/content → /home/ubuntu)
✅ /content/main.py enviado para /home/ubuntu/main.py
✅ Comando de reinício enviado
📋 Últimas linhas do log:
/home/ubuntu/venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ubuntu/venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ubun

# Vamos confirmar com um teste direto, esperando um pouco mais

In [13]:
import requests

resposta = requests.get("http://140.238.178.157:8000/")
print("Health check:", resposta.json())

resposta_perfil = requests.get("http://140.238.178.157:8000/explicabilidade/perfil")
print("\n📊 Explicabilidade — Perfil Financeiro:")
for item in resposta_perfil.json()["importancia_variaveis"][:3]:
    print(f"   {item['variavel']}: {item['importancia_percentual']}%")

resposta_categoria = requests.get("http://140.238.178.157:8000/explicabilidade/categoria?top_n=5")
print("\n📊 Explicabilidade — Categoria (exemplo 'entretenimento'):")
print(resposta_categoria.json()["palavras_por_categoria"].get("entretenimento"))

ConnectionError: HTTPConnectionPool(host='140.238.178.157', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7846589f1df0>: Failed to establish a new connection: [Errno 113] No route to host'))

Diagnóstico: qual versão do Pydantic está no **servidor**

In [14]:
# ============================================================
# Diagnóstico Completo — Comparação Local vs. Servidor
# ============================================================
"""
Compara o arquivo main.py local (Colab) com o arquivo main.py no
servidor de produção, verificando hash MD5 e presença da correção
'min_length', para identificar por que a validação de lista vazia
não está sendo aplicada em produção.
"""

import paramiko
import hashlib


def calcular_hash_local(caminho: str = '/content/main.py') -> tuple:
    """Lê o arquivo local e calcula seu hash MD5."""
    with open(caminho, 'r') as f:
        conteudo = f.read()
    hash_md5 = hashlib.md5(conteudo.encode()).hexdigest()
    return conteudo, hash_md5


def diagnosticar_servidor(ip_servidor: str, usuario: str, caminho_chave: str) -> dict:
    """Conecta ao servidor e coleta hash, tamanho e presença da correção no main.py remoto."""
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    comandos = {
        'hash': "md5sum /home/ubuntu/main.py | awk '{print $1}'",
        'tamanho': "ls -la /home/ubuntu/main.py | awk '{print $5}'",
        'min_length': "grep -c 'min_length' /home/ubuntu/main.py || echo 0",
        'data_modificacao': "stat -c '%y' /home/ubuntu/main.py",
        'processo_ativo': "ps aux | grep '[u]vicorn' | awk '{print $2}'",
    }

    resultados = {}
    for chave_resultado, comando in comandos.items():
        stdin, stdout, stderr = cliente_ssh.exec_command(comando)
        resultados[chave_resultado] = stdout.read().decode().strip()

    cliente_ssh.close()
    return resultados


def exibir_diagnostico(conteudo_local: str, hash_local: str, dados_servidor: dict) -> None:
    """Imprime a comparação lado a lado entre local e servidor."""
    print("=" * 60)
    print("📋 ARQUIVO LOCAL (Colab)")
    print("=" * 60)
    print(f"Hash MD5: {hash_local}")
    print(f"Tamanho: {len(conteudo_local)} caracteres")
    print(f"Contém 'min_length'? {'✅ Sim' if 'min_length' in conteudo_local else '❌ Não'}")

    print("\n" + "=" * 60)
    print("📋 ARQUIVO NO SERVIDOR")
    print("=" * 60)
    print(f"Hash MD5: {dados_servidor['hash']}")
    print(f"Tamanho: {dados_servidor['tamanho']} bytes")
    print(f"Ocorrências de 'min_length': {dados_servidor['min_length']}")
    print(f"Última modificação: {dados_servidor['data_modificacao']}")
    print(f"PID do processo uvicorn ativo: {dados_servidor['processo_ativo'] or '(nenhum processo encontrado)'}")

    print("\n" + "=" * 60)
    print("🔍 CONCLUSÃO")
    print("=" * 60)
    if hash_local == dados_servidor['hash']:
        print("✅ Os arquivos são IDÊNTICOS (mesmo hash).")
        if dados_servidor['min_length'] == '0':
            print("⚠️ Mas 'min_length' não está no arquivo — a correção não foi salva no CONTEUDO_MAIN antes de escrever o arquivo.")
        else:
            print("⚠️ A correção ESTÁ no arquivo. O processo ativo pode ser antigo (verificar PID x horário de início) ou o Pydantic não está aplicando a restrição.")
    else:
        print("❌ Os arquivos são DIFERENTES — o arquivo enviado ao servidor não é a versão mais recente do Colab.")


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
conteudo_local, hash_local = calcular_hash_local()
dados_servidor = diagnosticar_servidor(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
exibir_diagnostico(conteudo_local, hash_local, dados_servidor)

ModuleNotFoundError: No module named 'paramiko'

Célula única — atualiza a API com os dois novos endpoints + **testes**

In [15]:
# ============================================================
# Item 2 do Recurso Opcional — Explicabilidade dos Modelos
# Módulo: API + Testes, Célula Única e Autocontida (v3)
# ============================================================
"""
Adiciona dois endpoints de explicabilidade, sem alterar a lógica de
predição já validada:

    - GET /explicabilidade/perfil: importância de cada variável no
      Random Forest (Fase 6), extraída via feature_importances_.
    - GET /explicabilidade/categoria: palavras mais influentes na
      decisão de cada categoria, extraídas dos coeficientes da
      Regressão Logística (Fase 5) cruzados com o vocabulário do
      TfidfVectorizer.

Nenhum modelo é retreinado — a explicabilidade é extraída dos mesmos
artefatos já carregados em produção.
"""

import subprocess
import sys
import os


def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas")


CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao de transacoes + perfil financeiro + recomendacoes +
explicabilidade dos modelos, seguindo o contrato do edital do
Hackathon ONE (Alura + Oracle).
"""

import os
import joblib
import numpy as np
import pandas as pd
import requests
from contextlib import asynccontextmanager
from typing import Literal
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}
PASTA_MODELOS = "/content/modelos_api"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes: list, artefatos: dict) -> list:
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas: list) -> dict:
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas: list) -> dict:
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos) -> tuple:
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil: str, resumo_gastos: dict, frequencia_poupanca: str) -> list:
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada: dict, artefatos: dict) -> dict:
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": {k: v for k, v in resumo_gastos_edital.items() if v > 0},
        "recomendacoes": recomendacoes,
    }


def obter_importancia_variaveis_perfil(artefatos: dict) -> list:
    """Extrai a importancia de cada variavel do Random Forest (modelo de perfil).

    feature_importances_ e um atributo nativo de modelos baseados em
    arvore no scikit-learn — nao exige nenhum calculo adicional,
    apenas leitura do modelo ja treinado.
    """
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos: dict, top_n: int = 8) -> dict:
    """Extrai as palavras mais influentes na decisao de cada categoria.

    Para modelos lineares como a Regressao Logistica, o coeficiente de
    cada palavra (feature do TF-IDF) indica o quanto ela empurra a
    decisao em direcao aquela categoria — coeficientes mais altos
    (positivos) sao as palavras mais caracteristicas da classe.
    """
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_

    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    print("Modelos carregados com sucesso")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    return analisar_financas(dados.model_dump(), artefatos_globais)


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    """Retorna a importancia de cada variavel na decisao do modelo de perfil financeiro."""
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20, description="Quantidade de palavras por categoria")):
    """Retorna as palavras mais influentes na decisao de cada categoria de despesa."""
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}
'''


CONTEUDO_TESTES = '''
"""Suite de testes automatizados da API de Analise Financeira."""
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


class TestHealthCheck:
    def test_status_no_ar(self, cliente):
        resposta = cliente.get("/")
        assert resposta.status_code == 200
        assert resposta.json()["status"] == "API no ar"

    def test_modelos_carregados(self, cliente):
        resposta = cliente.get("/")
        assert resposta.json()["modelos_carregados"] is True


class TestAnaliseFinanceira:
    def test_perfil_saudavel(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}, {"descricao": "Streaming", "valor": 40}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Saudavel"

    def test_perfil_em_risco(self, cliente):
        dados = {"renda_mensal": 3000, "nivel_endividamento": 62, "frequencia_poupanca": "Baixa",
            "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}, {"descricao": "Posto Ipiranga", "valor": 250}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Em risco"

    def test_perfil_em_observacao(self, cliente):
        dados = {"renda_mensal": 5200, "nivel_endividamento": 38, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Mercado Atacadao", "valor": 380}, {"descricao": "Aplicativo Uber", "valor": 150}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Em observacao"

    def test_lista_transacoes_vazia_e_rejeitada(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media", "transacoes": []}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422


class TestExplicabilidade:
    """Testes dos novos endpoints de explicabilidade (Item 2 do bonus)."""

    def test_explicabilidade_perfil_retorna_variaveis(self, cliente):
        resposta = cliente.get("/explicabilidade/perfil")
        assert resposta.status_code == 200
        corpo = resposta.json()
        assert "importancia_variaveis" in corpo
        assert len(corpo["importancia_variaveis"]) == 11

    def test_explicabilidade_perfil_endividamento_e_mais_relevante(self, cliente):
        """Confirma que nivel_endividamento continua sendo a variavel mais importante,
        batendo com o que foi documentado na Fase 6 (69,8%)."""
        resposta = cliente.get("/explicabilidade/perfil")
        variaveis = resposta.json()["importancia_variaveis"]
        assert variaveis[0]["variavel"] == "nivel_endividamento"

    def test_explicabilidade_categoria_retorna_7_categorias(self, cliente):
        resposta = cliente.get("/explicabilidade/categoria")
        assert resposta.status_code == 200
        corpo = resposta.json()
        assert len(corpo["palavras_por_categoria"]) == 7

    def test_explicabilidade_categoria_respeita_top_n(self, cliente):
        resposta = cliente.get("/explicabilidade/categoria?top_n=3")
        corpo = resposta.json()
        for palavras in corpo["palavras_por_categoria"].values():
            assert len(palavras) == 3


class TestClassificarTransacoes:
    def test_classifica_transacoes_corretamente(self, cliente):
        dados = {"transacoes": [{"descricao": "Netflix", "valor": 40}, {"descricao": "Posto Ipiranga", "valor": 200}]}
        resposta = cliente.post("/classificar-transacoes", json=dados)
        assert resposta.status_code == 200
        corpo = resposta.json()
        assert len(corpo["transacoes_classificadas"]) == 2
'''


def escrever_arquivos() -> None:
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ main.py e test_main.py escritos em /content (com explicabilidade)")


def executar_suite_de_testes() -> None:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    if resultado.stderr:
        print("⚠️ Saída de erro/avisos:")
        print(resultado.stderr)


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
instalar_dependencias()
escrever_arquivos()
executar_suite_de_testes()

✅ Dependências instaladas
✅ main.py e test_main.py escritos em /content (com explicabilidade)
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 11 items

test_main.py::TestHealthCheck::test_status_no_ar PASSED                  [  9%]
test_main.py::TestHealthCheck::test_modelos_carregados PASSED            [ 18%]
test_main.py::TestAnaliseFinanceira::test_perfil_saudavel PASSED         [ 27%]
test_main.py::TestAnaliseFinanceira::test_perfil_em_risco PASSED         [ 36%]
test_main.py::TestAnaliseFinanceira::test_perfil_em_observacao PASSED    [ 45%]
test_main.py::TestAnaliseFinanceira::test_lista_transacoes_vazia_e_rejeitada PASSED [ 54%]
test_main.py::TestExplicabilidade::test_explicabilidade_perfil_retorna_variaveis PASSED [ 63%]
test_main.py::TestExplic

Célula única — já corrigindo o problema recorrente de caminho

In [16]:
# ============================================================
# Item 3 do Recurso Opcional — Histórico de Análises
# Módulo: API + Testes, Célula Única e Autocontida (v4)
# ============================================================
"""
Adiciona persistência de histórico via SQLite: cada chamada bem
sucedida ao endpoint /analise-financeira é automaticamente registrada
num banco de dados leve, sem necessidade de serviço externo — adequado
para a instância Always Free (1GB RAM).

Esta é a FUNDAÇÃO para os próximos itens do bônus: Alertas (Item 4),
Dashboard (Item 6) e Visualização de evolução (Item 7) vão consumir
este mesmo histórico.
"""

import subprocess
import sys
import os


def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas")


CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao de transacoes + perfil financeiro + recomendacoes +
explicabilidade + historico de analises (SQLite), seguindo o contrato
do edital do Hackathon ONE (Alura + Oracle).
"""

import os
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from contextlib import asynccontextmanager
from typing import Literal, Optional
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}

# --- CAMINHOS DO AMBIENTE ---
# ATENCAO: estes caminhos precisam apontar para o SERVIDOR (/home/ubuntu/...)
# antes de cada deploy. A celula de deploy corrige isso automaticamente,
# mas ao editar este template diretamente, confira este bloco primeiro.
PASTA_MODELOS = "/home/ubuntu/modelos_api"
CAMINHO_BANCO_DADOS = "/home/ubuntu/historico.db"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes: list, artefatos: dict) -> list:
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas: list) -> dict:
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas: list) -> dict:
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos) -> tuple:
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil: str, resumo_gastos: dict, frequencia_poupanca: str) -> list:
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada: dict, artefatos: dict) -> dict:
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": {k: v for k, v in resumo_gastos_edital.items() if v > 0},
        "recomendacoes": recomendacoes,
    }


def obter_importancia_variaveis_perfil(artefatos: dict) -> list:
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos: dict, top_n: int = 8) -> dict:
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_
    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


# ------------------------------------------------------------
# HISTORICO DE ANALISES (SQLite) - Item 3 do bonus
# ------------------------------------------------------------

def inicializar_banco_de_dados(caminho_db: str) -> None:
    """Cria a tabela de historico, caso ainda nao exista.

    idempotente: pode ser chamada toda vez que a API sobe, sem
    duplicar ou apagar dados ja existentes (CREATE TABLE IF NOT EXISTS).
    """
    conexao = sqlite3.connect(caminho_db)
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS analises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_hora TEXT NOT NULL,
            renda_mensal REAL NOT NULL,
            nivel_endividamento REAL NOT NULL,
            frequencia_poupanca TEXT NOT NULL,
            transacoes_json TEXT NOT NULL,
            perfil_financeiro TEXT NOT NULL,
            probabilidade REAL NOT NULL,
            resumo_gastos_json TEXT NOT NULL,
            recomendacoes_json TEXT NOT NULL
        )
    """)
    conexao.commit()
    conexao.close()


def salvar_analise_no_historico(caminho_db: str, dados_entrada: dict, resultado: dict) -> int:
    """Grava uma analise concluida no historico e retorna o ID gerado."""
    conexao = sqlite3.connect(caminho_db)
    cursor = conexao.execute(
        """INSERT INTO analises
           (data_hora, renda_mensal, nivel_endividamento, frequencia_poupanca,
            transacoes_json, perfil_financeiro, probabilidade, resumo_gastos_json, recomendacoes_json)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(),
            dados_entrada["renda_mensal"],
            dados_entrada["nivel_endividamento"],
            dados_entrada["frequencia_poupanca"],
            json.dumps(dados_entrada["transacoes"], ensure_ascii=False),
            resultado["perfil_financeiro"],
            resultado["probabilidade"],
            json.dumps(resultado["resumo_gastos"], ensure_ascii=False),
            json.dumps(resultado["recomendacoes"], ensure_ascii=False),
        )
    )
    conexao.commit()
    id_gerado = cursor.lastrowid
    conexao.close()
    return id_gerado


def listar_historico(caminho_db: str, limite: int = 20) -> list:
    """Lista as analises mais recentes, da mais nova para a mais antiga."""
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linhas = conexao.execute(
        "SELECT * FROM analises ORDER BY id DESC LIMIT ?", (limite,)
    ).fetchall()
    conexao.close()
    return [_linha_para_dict(linha) for linha in linhas]


def obter_analise_por_id(caminho_db: str, id_analise: int) -> Optional[dict]:
    """Busca uma analise especifica pelo ID, ou None se nao existir."""
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linha = conexao.execute("SELECT * FROM analises WHERE id = ?", (id_analise,)).fetchone()
    conexao.close()
    return _linha_para_dict(linha) if linha else None


def _linha_para_dict(linha: sqlite3.Row) -> dict:
    """Converte uma linha do banco em dict, desserializando os campos JSON."""
    return {
        "id": linha["id"],
        "data_hora": linha["data_hora"],
        "renda_mensal": linha["renda_mensal"],
        "nivel_endividamento": linha["nivel_endividamento"],
        "frequencia_poupanca": linha["frequencia_poupanca"],
        "transacoes": json.loads(linha["transacoes_json"]),
        "perfil_financeiro": linha["perfil_financeiro"],
        "probabilidade": linha["probabilidade"],
        "resumo_gastos": json.loads(linha["resumo_gastos_json"]),
        "recomendacoes": json.loads(linha["recomendacoes_json"]),
    }


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    inicializar_banco_de_dados(CAMINHO_BANCO_DADOS)
    print("Modelos carregados e banco de historico pronto")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_dict, resultado)
    return {**resultado, "id_historico": id_historico}


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20)):
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}


@app.get("/historico")
def historico(limite: int = Query(default=20, ge=1, le=100, description="Quantidade maxima de registros")):
    """Lista as analises mais recentes ja realizadas pela API."""
    return {"total_retornado": None, "analises": listar_historico(CAMINHO_BANCO_DADOS, limite)}


@app.get("/historico/{id_analise}")
def historico_por_id(id_analise: int):
    """Retorna o detalhe de uma analise especifica pelo ID."""
    analise = obter_analise_por_id(CAMINHO_BANCO_DADOS, id_analise)
    if analise is None:
        raise HTTPException(status_code=404, detail=f"Analise com id {id_analise} nao encontrada")
    return analise
'''


CONTEUDO_TESTES = '''
"""Suite de testes automatizados da API de Analise Financeira."""
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


class TestHealthCheck:
    def test_status_no_ar(self, cliente):
        resposta = cliente.get("/")
        assert resposta.status_code == 200
        assert resposta.json()["status"] == "API no ar"


class TestAnaliseFinanceira:
    def test_perfil_saudavel_e_salva_no_historico(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}, {"descricao": "Streaming", "valor": 40}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        corpo = resposta.json()
        assert corpo["perfil_financeiro"] == "Saudavel"
        assert "id_historico" in corpo
        assert isinstance(corpo["id_historico"], int)

    def test_lista_transacoes_vazia_e_rejeitada(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media", "transacoes": []}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422


class TestExplicabilidade:
    def test_explicabilidade_perfil_retorna_variaveis(self, cliente):
        resposta = cliente.get("/explicabilidade/perfil")
        assert resposta.status_code == 200
        assert len(resposta.json()["importancia_variaveis"]) == 11


class TestHistorico:
    """Testes do novo modulo de historico de analises (Item 3 do bonus)."""

    def test_historico_lista_analises_realizadas(self, cliente):
        dados = {"renda_mensal": 3000, "nivel_endividamento": 62, "frequencia_poupanca": "Baixa",
            "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}]}
        cliente.post("/analise-financeira", json=dados)

        resposta = cliente.get("/historico")
        assert resposta.status_code == 200
        assert len(resposta.json()["analises"]) >= 1

    def test_historico_respeita_limite(self, cliente):
        resposta = cliente.get("/historico?limite=1")
        assert resposta.status_code == 200
        assert len(resposta.json()["analises"]) == 1

    def test_historico_por_id_encontra_analise_criada(self, cliente):
        dados = {"renda_mensal": 5200, "nivel_endividamento": 38, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Curso Online Alura", "valor": 90}]}
        resposta_criacao = cliente.post("/analise-financeira", json=dados)
        id_criado = resposta_criacao.json()["id_historico"]

        resposta_busca = cliente.get(f"/historico/{id_criado}")
        assert resposta_busca.status_code == 200
        assert resposta_busca.json()["perfil_financeiro"] == "Em observacao"

    def test_historico_id_inexistente_retorna_404(self, cliente):
        resposta = cliente.get("/historico/999999")
        assert resposta.status_code == 404

    def test_historico_registro_contem_todos_os_campos(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Netflix", "valor": 40}]}
        id_criado = cliente.post("/analise-financeira", json=dados).json()["id_historico"]

        registro = cliente.get(f"/historico/{id_criado}").json()
        campos_esperados = ["id", "data_hora", "renda_mensal", "nivel_endividamento",
                             "frequencia_poupanca", "transacoes", "perfil_financeiro",
                             "probabilidade", "resumo_gastos", "recomendacoes"]
        for campo in campos_esperados:
            assert campo in registro
'''


def escrever_arquivos() -> None:
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ main.py e test_main.py escritos em /content (com histórico de análises)")


def executar_suite_de_testes() -> None:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    if resultado.stderr:
        print("⚠️ Saída de erro/avisos:")
        print(resultado.stderr)


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
instalar_dependencias()
escrever_arquivos()
executar_suite_de_testes()

✅ Dependências instaladas
✅ main.py e test_main.py escritos em /content (com histórico de análises)
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 9 items

test_main.py::TestHealthCheck::test_status_no_ar PASSED                  [ 11%]
test_main.py::TestAnaliseFinanceira::test_perfil_saudavel_e_salva_no_historico PASSED [ 22%]
test_main.py::TestAnaliseFinanceira::test_lista_transacoes_vazia_e_rejeitada PASSED [ 33%]
test_main.py::TestExplicabilidade::test_explicabilidade_perfil_retorna_variaveis PASSED [ 44%]
test_main.py::TestHistorico::test_historico_lista_analises_realizadas PASSED [ 55%]
test_main.py::TestHistorico::test_historico_respeita_limite PASSED       [ 66%]
test_main.py::TestHistorico::test_historico_por_id_encontra_analise_criada PASSED 

# Teste de histórico na API pública

In [17]:
import requests

dados = {
    "renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
    "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}]
}
resposta = requests.post("http://140.238.178.157:8000/analise-financeira", json=dados)
print("Análise criada:", resposta.json())

resposta_historico = requests.get("http://140.238.178.157:8000/historico")
print("\nHistórico (últimas análises):")
print(resposta_historico.json())

ConnectionError: HTTPConnectionPool(host='140.238.178.157', port=8000): Max retries exceeded with url: /analise-financeira (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x784658a15c10>: Failed to establish a new connection: [Errno 113] No route to host'))

TESTE  célula de diagnóstico rápido

In [18]:
import paramiko

chave = paramiko.RSAKey.from_private_key_file("ssh-key-2026-08-01.key")
cliente_ssh = paramiko.SSHClient()
cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
cliente_ssh.connect(hostname="140.238.178.157", username="ubuntu", pkey=chave, timeout=15)

stdin, stdout, stderr = cliente_ssh.exec_command("grep -c 'historico' /home/ubuntu/main.py")
print("Ocorrências de 'historico' no main.py do servidor:", stdout.read().decode().strip())

stdin, stdout, stderr = cliente_ssh.exec_command("stat -c '%y' /home/ubuntu/main.py")
print("Última modificação do arquivo:", stdout.read().decode().strip())

stdin, stdout, stderr = cliente_ssh.exec_command("ps aux | grep '[u]vicorn'")
print("Processo ativo:", stdout.read().decode().strip())

cliente_ssh.close()

ModuleNotFoundError: No module named 'paramiko'

Top Mega - Blaster

In [19]:
# ============================================================
# Item 3 do Recurso Opcional — Histórico de Análises
# Módulo Completo: API + Testes + Deploy + Confirmação em Produção
# Célula única, definitiva e autocontida
# ============================================================
"""
Fluxo completo numa única execução: escreve a API com histórico de
análises (SQLite), roda a suíte de testes local, envia para o servidor
de produção, reinicia a API remota, e confirma o funcionamento direto
na API pública — eliminando o risco de esquecer de rodar uma próxima
célula separada.
"""

import subprocess
import sys
import os
import time
import paramiko
import requests


# ------------------------------------------------------------
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
# ------------------------------------------------------------

def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas")


# ------------------------------------------------------------
# 2. CONTEÚDO DA API (com histórico de análises)
# ------------------------------------------------------------

CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao de transacoes + perfil financeiro + recomendacoes +
explicabilidade + historico de analises (SQLite).
"""

import os
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from contextlib import asynccontextmanager
from typing import Literal, Optional
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1)


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1)


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}

PASTA_MODELOS = "/home/ubuntu/modelos_api"
CAMINHO_BANCO_DADOS = "/home/ubuntu/historico.db"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes, artefatos):
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas):
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas):
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos):
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil, resumo_gastos, frequencia_poupanca):
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada, artefatos):
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": {k: v for k, v in resumo_gastos_edital.items() if v > 0},
        "recomendacoes": recomendacoes,
    }


def obter_importancia_variaveis_perfil(artefatos):
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos, top_n=8):
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_
    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


def inicializar_banco_de_dados(caminho_db):
    conexao = sqlite3.connect(caminho_db)
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS analises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_hora TEXT NOT NULL,
            renda_mensal REAL NOT NULL,
            nivel_endividamento REAL NOT NULL,
            frequencia_poupanca TEXT NOT NULL,
            transacoes_json TEXT NOT NULL,
            perfil_financeiro TEXT NOT NULL,
            probabilidade REAL NOT NULL,
            resumo_gastos_json TEXT NOT NULL,
            recomendacoes_json TEXT NOT NULL
        )
    """)
    conexao.commit()
    conexao.close()


def salvar_analise_no_historico(caminho_db, dados_entrada, resultado):
    conexao = sqlite3.connect(caminho_db)
    cursor = conexao.execute(
        """INSERT INTO analises
           (data_hora, renda_mensal, nivel_endividamento, frequencia_poupanca,
            transacoes_json, perfil_financeiro, probabilidade, resumo_gastos_json, recomendacoes_json)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(),
            dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"], dados_entrada["frequencia_poupanca"],
            json.dumps(dados_entrada["transacoes"], ensure_ascii=False),
            resultado["perfil_financeiro"], resultado["probabilidade"],
            json.dumps(resultado["resumo_gastos"], ensure_ascii=False),
            json.dumps(resultado["recomendacoes"], ensure_ascii=False),
        )
    )
    conexao.commit()
    id_gerado = cursor.lastrowid
    conexao.close()
    return id_gerado


def listar_historico(caminho_db, limite=20):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linhas = conexao.execute("SELECT * FROM analises ORDER BY id DESC LIMIT ?", (limite,)).fetchall()
    conexao.close()
    return [_linha_para_dict(linha) for linha in linhas]


def obter_analise_por_id(caminho_db, id_analise):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linha = conexao.execute("SELECT * FROM analises WHERE id = ?", (id_analise,)).fetchone()
    conexao.close()
    return _linha_para_dict(linha) if linha else None


def _linha_para_dict(linha):
    return {
        "id": linha["id"], "data_hora": linha["data_hora"],
        "renda_mensal": linha["renda_mensal"], "nivel_endividamento": linha["nivel_endividamento"],
        "frequencia_poupanca": linha["frequencia_poupanca"],
        "transacoes": json.loads(linha["transacoes_json"]),
        "perfil_financeiro": linha["perfil_financeiro"], "probabilidade": linha["probabilidade"],
        "resumo_gastos": json.loads(linha["resumo_gastos_json"]),
        "recomendacoes": json.loads(linha["recomendacoes_json"]),
    }


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    inicializar_banco_de_dados(CAMINHO_BANCO_DADOS)
    print("Modelos carregados e banco de historico pronto")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_dict, resultado)
    return {**resultado, "id_historico": id_historico}


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20)):
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}


@app.get("/historico")
def historico(limite: int = Query(default=20, ge=1, le=100)):
    return {"analises": listar_historico(CAMINHO_BANCO_DADOS, limite)}


@app.get("/historico/{id_analise}")
def historico_por_id(id_analise: int):
    analise = obter_analise_por_id(CAMINHO_BANCO_DADOS, id_analise)
    if analise is None:
        raise HTTPException(status_code=404, detail=f"Analise com id {id_analise} nao encontrada")
    return analise
'''


CONTEUDO_TESTES = '''
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


def test_status_no_ar(cliente):
    assert cliente.get("/").status_code == 200


def test_analise_salva_no_historico(cliente):
    dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
        "transacoes": [{"descricao": "Supermercado", "valor": 420}]}
    resposta = cliente.post("/analise-financeira", json=dados)
    assert resposta.status_code == 200
    assert "id_historico" in resposta.json()


def test_historico_lista(cliente):
    assert cliente.get("/historico").status_code == 200


def test_historico_por_id(cliente):
    dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
        "transacoes": [{"descricao": "Netflix", "valor": 40}]}
    id_criado = cliente.post("/analise-financeira", json=dados).json()["id_historico"]
    assert cliente.get(f"/historico/{id_criado}").status_code == 200


def test_historico_404(cliente):
    assert cliente.get("/historico/999999").status_code == 404
'''


def escrever_arquivos() -> None:
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ Arquivos escritos localmente")


def executar_suite_de_testes() -> bool:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    return resultado.returncode == 0


# ------------------------------------------------------------
# 3. DEPLOY PARA PRODUÇÃO
# ------------------------------------------------------------

def enviar_e_reiniciar(ip_servidor: str, usuario: str, caminho_chave: str) -> None:
    """Envia o main.py e reinicia a API, tudo numa única conexão SSH."""
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    sftp = cliente_ssh.open_sftp()
    sftp.put("/content/main.py", "/home/ubuntu/main.py")
    sftp.close()
    print("✅ main.py enviado para o servidor")

    cliente_ssh.exec_command("pkill -f uvicorn")
    time.sleep(2)

    comando_subir = (
        "cd /home/ubuntu && source venv/bin/activate && "
        "nohup uvicorn main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 < /dev/null &"
    )
    cliente_ssh.exec_command(comando_subir)
    time.sleep(1)
    cliente_ssh.close()
    print("✅ API reiniciada no servidor")


def confirmar_producao(ip_servidor: str) -> None:
    """Testa a API pública diretamente via HTTP, o critério mais confiável de sucesso."""
    time.sleep(5)
    try:
        resposta_saude = requests.get(f"http://{ip_servidor}:8000/", timeout=10)
        print("🔍 Health check:", resposta_saude.json())

        dados_teste = {
            "renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}]
        }
        resposta_analise = requests.post(f"http://{ip_servidor}:8000/analise-financeira", json=dados_teste, timeout=10)
        corpo_analise = resposta_analise.json()
        print("\n🔍 Análise de teste:", corpo_analise)

        if "id_historico" in corpo_analise:
            print("\n🎉 Item 3 (Histórico) CONFIRMADO em produção!")
            resposta_historico = requests.get(f"http://{ip_servidor}:8000/historico", timeout=10)
            print("📋 Histórico:", resposta_historico.json())
        else:
            print("\n⚠️ 'id_historico' ainda não aparece na resposta — deploy pode não ter concluído")

    except Exception as e:
        print(f"❌ Erro ao testar produção: {e}")


# ------------------------------------------------------------
# EXECUÇÃO COMPLETA
# ------------------------------------------------------------
IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

instalar_dependencias()
escrever_arquivos()
testes_ok = executar_suite_de_testes()

if testes_ok and os.path.exists(CAMINHO_CHAVE):
    print("\n" + "=" * 60)
    print("🚀 Testes locais OK — iniciando deploy para produção")
    print("=" * 60)
    enviar_e_reiniciar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
    confirmar_producao(IP_SERVIDOR)
elif not testes_ok:
    print("\n❌ Testes locais falharam — deploy NÃO realizado, corrija antes de continuar")
else:
    print(f"\n⚠️ Chave '{CAMINHO_CHAVE}' não encontrada — faça upload antes: from google.colab import files; files.upload()")

ModuleNotFoundError: No module named 'paramiko'

In [20]:
import requests
import time

time.sleep(3)

resposta_saude = requests.get("http://140.238.178.157:8000/", timeout=10)
print("🔍 Health check:", resposta_saude.json())

dados_teste = {
    "renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
    "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}]
}
resposta_analise = requests.post("http://140.238.178.157:8000/analise-financeira", json=dados_teste, timeout=10)
corpo = resposta_analise.json()
print("\n🔍 Análise de teste:", corpo)

if "id_historico" in corpo:
    print("\n🎉 Item 3 (Histórico) CONFIRMADO em produção!")
    resposta_historico = requests.get("http://140.238.178.157:8000/historico", timeout=10)
    print("📋 Histórico:", resposta_historico.json())
else:
    print("\n⚠️ 'id_historico' ainda não aparece — algo não subiu corretamente")

ConnectionError: HTTPConnectionPool(host='140.238.178.157', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x784659c6d010>: Failed to establish a new connection: [Errno 113] No route to host'))

Célula única — Item 4 ( Alerta de Gastos elevados ) completo (segue o mesmo padrão: escreve → testa → deploy → confirma)

In [21]:
# ============================================================
# Item 4 do Recurso Opcional — Alertas de Gastos Elevados
# Módulo Completo: API + Testes + Deploy + Confirmação em Produção
# ============================================================
"""
Adiciona alertas automáticos quando uma categoria de gasto ultrapassa
30% da renda mensal, ou quando o comprometimento total ultrapassa 90%
— reaproveitando os mesmos limiares já usados na regra de perfil
financeiro (Fase 6), mantendo consistência de critério em todo o projeto.
"""

import subprocess
import sys
import os
import time
import paramiko
import requests


def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas")


CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao + perfil + recomendacoes + explicabilidade + historico +
alertas de gastos elevados.
"""

import os
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from contextlib import asynccontextmanager
from typing import Literal, Optional
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1)


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1)


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

# Limiares de alerta - consistentes com os limiares da regra de perfil (Fase 6)
LIMIAR_CATEGORIA_ELEVADA = 0.30  # 30% da renda numa unica categoria
LIMIAR_COMPROMETIMENTO_TOTAL = 0.90  # 90% da renda no total

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}

PASTA_MODELOS = "/home/ubuntu/modelos_api"
CAMINHO_BANCO_DADOS = "/home/ubuntu/historico.db"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes, artefatos):
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas):
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas):
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos):
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil, resumo_gastos, frequencia_poupanca):
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def gerar_alertas_gastos_elevados(renda_mensal: float, resumo_gastos: dict) -> list:
    """Gera alertas quando um gasto (por categoria ou total) e considerado elevado.

    Usa os mesmos limiares conceituais da regra de perfil financeiro
    (Fase 6), aplicados agora por categoria individual, nao so no total.
    Isso da ao usuario um diagnostico mais granular: mesmo com perfil
    'Saudavel' no geral, uma categoria especifica pode merecer atencao.
    """
    alertas = []

    for categoria, valor in resumo_gastos.items():
        percentual = valor / renda_mensal
        if percentual >= LIMIAR_CATEGORIA_ELEVADA:
            alertas.append({
                "tipo": "categoria_elevada",
                "categoria": categoria,
                "valor": valor,
                "percentual_da_renda": round(percentual * 100, 1),
                "mensagem": f"Gasto com {categoria} representa {round(percentual * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_CATEGORIA_ELEVADA * 100)}%)",
            })

    comprometimento_total = sum(resumo_gastos.values()) / renda_mensal
    if comprometimento_total >= LIMIAR_COMPROMETIMENTO_TOTAL:
        alertas.append({
            "tipo": "comprometimento_total_elevado",
            "categoria": None,
            "valor": round(sum(resumo_gastos.values()), 2),
            "percentual_da_renda": round(comprometimento_total * 100, 1),
            "mensagem": f"Gasto total representa {round(comprometimento_total * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_COMPROMETIMENTO_TOTAL * 100)}%)",
        })

    return alertas


def analisar_financas(dados_entrada, artefatos):
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    resumo_gastos_filtrado = {k: v for k, v in resumo_gastos_edital.items() if v > 0}
    alertas = gerar_alertas_gastos_elevados(dados_entrada["renda_mensal"], resumo_gastos_filtrado)
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": resumo_gastos_filtrado,
        "recomendacoes": recomendacoes,
        "alertas": alertas,
    }


def obter_importancia_variaveis_perfil(artefatos):
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos, top_n=8):
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_
    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


def inicializar_banco_de_dados(caminho_db):
    conexao = sqlite3.connect(caminho_db)
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS analises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_hora TEXT NOT NULL,
            renda_mensal REAL NOT NULL,
            nivel_endividamento REAL NOT NULL,
            frequencia_poupanca TEXT NOT NULL,
            transacoes_json TEXT NOT NULL,
            perfil_financeiro TEXT NOT NULL,
            probabilidade REAL NOT NULL,
            resumo_gastos_json TEXT NOT NULL,
            recomendacoes_json TEXT NOT NULL,
            alertas_json TEXT NOT NULL DEFAULT "[]"
        )
    """)
    # Garante compatibilidade se a tabela ja existia sem a coluna alertas_json
    try:
        conexao.execute("ALTER TABLE analises ADD COLUMN alertas_json TEXT NOT NULL DEFAULT \\'[]\\'")
    except sqlite3.OperationalError:
        pass  # coluna ja existe
    conexao.commit()
    conexao.close()


def salvar_analise_no_historico(caminho_db, dados_entrada, resultado):
    conexao = sqlite3.connect(caminho_db)
    cursor = conexao.execute(
        """INSERT INTO analises
           (data_hora, renda_mensal, nivel_endividamento, frequencia_poupanca,
            transacoes_json, perfil_financeiro, probabilidade, resumo_gastos_json,
            recomendacoes_json, alertas_json)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(),
            dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"], dados_entrada["frequencia_poupanca"],
            json.dumps(dados_entrada["transacoes"], ensure_ascii=False),
            resultado["perfil_financeiro"], resultado["probabilidade"],
            json.dumps(resultado["resumo_gastos"], ensure_ascii=False),
            json.dumps(resultado["recomendacoes"], ensure_ascii=False),
            json.dumps(resultado["alertas"], ensure_ascii=False),
        )
    )
    conexao.commit()
    id_gerado = cursor.lastrowid
    conexao.close()
    return id_gerado


def listar_historico(caminho_db, limite=20):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linhas = conexao.execute("SELECT * FROM analises ORDER BY id DESC LIMIT ?", (limite,)).fetchall()
    conexao.close()
    return [_linha_para_dict(linha) for linha in linhas]


def obter_analise_por_id(caminho_db, id_analise):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linha = conexao.execute("SELECT * FROM analises WHERE id = ?", (id_analise,)).fetchone()
    conexao.close()
    return _linha_para_dict(linha) if linha else None


def _linha_para_dict(linha):
    chaves = linha.keys()
    return {
        "id": linha["id"], "data_hora": linha["data_hora"],
        "renda_mensal": linha["renda_mensal"], "nivel_endividamento": linha["nivel_endividamento"],
        "frequencia_poupanca": linha["frequencia_poupanca"],
        "transacoes": json.loads(linha["transacoes_json"]),
        "perfil_financeiro": linha["perfil_financeiro"], "probabilidade": linha["probabilidade"],
        "resumo_gastos": json.loads(linha["resumo_gastos_json"]),
        "recomendacoes": json.loads(linha["recomendacoes_json"]),
        "alertas": json.loads(linha["alertas_json"]) if "alertas_json" in chaves and linha["alertas_json"] else [],
    }


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    inicializar_banco_de_dados(CAMINHO_BANCO_DADOS)
    print("Modelos carregados e banco de historico pronto")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_dict, resultado)
    return {**resultado, "id_historico": id_historico}


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20)):
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}


@app.get("/historico")
def historico(limite: int = Query(default=20, ge=1, le=100)):
    return {"analises": listar_historico(CAMINHO_BANCO_DADOS, limite)}


@app.get("/historico/{id_analise}")
def historico_por_id(id_analise: int):
    analise = obter_analise_por_id(CAMINHO_BANCO_DADOS, id_analise)
    if analise is None:
        raise HTTPException(status_code=404, detail=f"Analise com id {id_analise} nao encontrada")
    return analise
'''


CONTEUDO_TESTES = '''
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


def test_status_no_ar(cliente):
    assert cliente.get("/").status_code == 200


def test_sem_alertas_quando_gastos_equilibrados(cliente):
    """Gastos bem distribuidos, nenhum acima de 30% da renda, nao devem gerar alerta."""
    dados = {"renda_mensal": 10000, "nivel_endividamento": 10, "frequencia_poupanca": "Alta",
        "transacoes": [{"descricao": "Supermercado", "valor": 500}, {"descricao": "Streaming", "valor": 40}]}
    resposta = cliente.post("/analise-financeira", json=dados)
    assert resposta.status_code == 200
    assert resposta.json()["alertas"] == []


def test_alerta_categoria_elevada(cliente):
    """Um gasto de 1200 numa renda de 3000 (40%) deve gerar alerta de categoria elevada."""
    dados = {"renda_mensal": 3000, "nivel_endividamento": 20, "frequencia_poupanca": "Media",
        "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}]}
    resposta = cliente.post("/analise-financeira", json=dados)
    assert resposta.status_code == 200
    alertas = resposta.json()["alertas"]
    assert any(a["tipo"] == "categoria_elevada" for a in alertas)


def test_alerta_comprometimento_total(cliente):
    """Gastos somando mais de 90% da renda devem gerar alerta de comprometimento total."""
    dados = {"renda_mensal": 2000, "nivel_endividamento": 20, "frequencia_poupanca": "Baixa",
        "transacoes": [
            {"descricao": "Aluguel Residencial", "valor": 900},
            {"descricao": "Supermercado", "valor": 700},
            {"descricao": "Posto Ipiranga", "valor": 350},
        ]}
    resposta = cliente.post("/analise-financeira", json=dados)
    assert resposta.status_code == 200
    alertas = resposta.json()["alertas"]
    assert any(a["tipo"] == "comprometimento_total_elevado" for a in alertas)


def test_alerta_persiste_no_historico(cliente):
    dados = {"renda_mensal": 3000, "nivel_endividamento": 20, "frequencia_poupanca": "Media",
        "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}]}
    id_criado = cliente.post("/analise-financeira", json=dados).json()["id_historico"]
    registro = cliente.get(f"/historico/{id_criado}").json()
    assert "alertas" in registro
    assert len(registro["alertas"]) >= 1
'''


def escrever_arquivos() -> None:
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ Arquivos escritos localmente")


def executar_suite_de_testes() -> bool:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    return resultado.returncode == 0


def enviar_e_reiniciar(ip_servidor: str, usuario: str, caminho_chave: str) -> None:
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    sftp = cliente_ssh.open_sftp()
    sftp.put("/content/main.py", "/home/ubuntu/main.py")
    sftp.close()
    print("✅ main.py enviado para o servidor")

    cliente_ssh.exec_command("pkill -f uvicorn")
    time.sleep(2)

    comando_subir = (
        "cd /home/ubuntu && source venv/bin/activate && "
        "nohup uvicorn main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 < /dev/null &"
    )
    cliente_ssh.exec_command(comando_subir)
    time.sleep(1)
    cliente_ssh.close()
    print("✅ API reiniciada no servidor")


def confirmar_producao(ip_servidor: str) -> None:
    time.sleep(5)
    try:
        resposta_saude = requests.get(f"http://{ip_servidor}:8000/", timeout=10)
        print("🔍 Health check:", resposta_saude.json())

        dados_teste = {
            "renda_mensal": 3000, "nivel_endividamento": 20, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}]
        }
        resposta_analise = requests.post(f"http://{ip_servidor}:8000/analise-financeira", json=dados_teste, timeout=10)
        corpo = resposta_analise.json()
        print("\n🔍 Análise de teste (deve gerar alerta de categoria elevada):")
        print(corpo)

        if "alertas" in corpo and len(corpo["alertas"]) > 0:
            print("\n🎉 Item 4 (Alertas) CONFIRMADO em produção!")
        else:
            print("\n⚠️ Campo 'alertas' ausente ou vazio — deploy pode não ter concluído")

    except Exception as e:
        print(f"❌ Erro ao testar produção: {e}")


# ------------------------------------------------------------
# EXECUÇÃO COMPLETA
# ------------------------------------------------------------
IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

instalar_dependencias()
escrever_arquivos()
testes_ok = executar_suite_de_testes()

if testes_ok and os.path.exists(CAMINHO_CHAVE):
    print("\n" + "=" * 60)
    print("🚀 Testes locais OK — iniciando deploy para produção")
    print("=" * 60)
    enviar_e_reiniciar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
    confirmar_producao(IP_SERVIDOR)
elif not testes_ok:
    print("\n❌ Testes locais falharam — deploy NÃO realizado, corrija antes de continuar")
else:
    print(f"\n⚠️ Chave '{CAMINHO_CHAVE}' não encontrada — faça upload antes: from google.colab import files; files.upload()")

ModuleNotFoundError: No module named 'paramiko'

Célula única — Item 5 completo Processamento em Lote via CSV

In [22]:
# ============================================================
# Item 5 do Recurso Opcional — Processamento em Lote via CSV
# Módulo Completo: API + Testes + Deploy + Confirmação em Produção
# ============================================================
"""
Adiciona um endpoint que aceita upload de arquivo CSV com múltiplas
transações de múltiplos usuários (agrupadas por usuario_id), processa
cada usuário como uma análise financeira completa, e salva todas no
histórico — reaproveitando 100% da lógica de análise já validada.
"""

import subprocess
import sys
import os
import time
import io
import paramiko
import requests


def instalar_dependencias() -> None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "python-multipart", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas (incluindo python-multipart, necessário para upload de arquivos)")


CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao + perfil + recomendacoes + explicabilidade + historico +
alertas + processamento em lote via CSV.
"""

import os
import json
import sqlite3
import io
import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from contextlib import asynccontextmanager
from typing import Literal, Optional
from fastapi import FastAPI, HTTPException, Query, UploadFile, File
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1)


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1)


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

FEATURES_MODELO_PERFIL = [
    "renda_mensal", "nivel_endividamento", "frequencia_poupanca_cod", "comprometimento_gastos",
    "Alimentacao", "Moradia", "Transporte", "Saude", "Educacao", "Lazer", "Servicos",
]

LIMIAR_CATEGORIA_ELEVADA = 0.30
LIMIAR_COMPROMETIMENTO_TOTAL = 0.90

COLUNAS_CSV_OBRIGATORIAS = ["usuario_id", "renda_mensal", "nivel_endividamento", "frequencia_poupanca", "descricao", "valor"]

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}

PASTA_MODELOS = "/home/ubuntu/modelos_api"
CAMINHO_BANCO_DADOS = "/home/ubuntu/historico.db"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes, artefatos):
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas):
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas):
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos):
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil, resumo_gastos, frequencia_poupanca):
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def gerar_alertas_gastos_elevados(renda_mensal, resumo_gastos):
    alertas = []
    for categoria, valor in resumo_gastos.items():
        percentual = valor / renda_mensal
        if percentual >= LIMIAR_CATEGORIA_ELEVADA:
            alertas.append({
                "tipo": "categoria_elevada", "categoria": categoria, "valor": valor,
                "percentual_da_renda": round(percentual * 100, 1),
                "mensagem": f"Gasto com {categoria} representa {round(percentual * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_CATEGORIA_ELEVADA * 100)}%)",
            })
    comprometimento_total = sum(resumo_gastos.values()) / renda_mensal
    if comprometimento_total >= LIMIAR_COMPROMETIMENTO_TOTAL:
        alertas.append({
            "tipo": "comprometimento_total_elevado", "categoria": None,
            "valor": round(sum(resumo_gastos.values()), 2), "percentual_da_renda": round(comprometimento_total * 100, 1),
            "mensagem": f"Gasto total representa {round(comprometimento_total * 100, 1)}% da renda mensal (limiar: {int(LIMIAR_COMPROMETIMENTO_TOTAL * 100)}%)",
        })
    return alertas


def analisar_financas(dados_entrada, artefatos):
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    resumo_gastos_filtrado = {k: v for k, v in resumo_gastos_edital.items() if v > 0}
    alertas = gerar_alertas_gastos_elevados(dados_entrada["renda_mensal"], resumo_gastos_filtrado)
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": resumo_gastos_filtrado, "recomendacoes": recomendacoes, "alertas": alertas,
    }


def obter_importancia_variaveis_perfil(artefatos):
    modelo = artefatos["modelo_perfil"]
    importancias = modelo.feature_importances_
    pares = list(zip(FEATURES_MODELO_PERFIL, importancias))
    pares_ordenados = sorted(pares, key=lambda x: x[1], reverse=True)
    return [{"variavel": nome, "importancia_percentual": round(float(valor) * 100, 2)} for nome, valor in pares_ordenados]


def obter_palavras_influentes_categoria(artefatos, top_n=8):
    modelo = artefatos["modelo_categoria"]
    vocabulario = artefatos["vetorizador_tfidf"].get_feature_names_out()
    classes = artefatos["codificador_categorias"].classes_
    resultado = {}
    for indice_classe, nome_classe_interno in enumerate(classes):
        coeficientes = modelo.coef_[indice_classe]
        indices_top = np.argsort(coeficientes)[::-1][:top_n]
        nome_edital = NOMES_EDITAL.get(nome_classe_interno, nome_classe_interno.lower())
        resultado[nome_edital] = [vocabulario[i] for i in indices_top]
    return resultado


def inicializar_banco_de_dados(caminho_db):
    conexao = sqlite3.connect(caminho_db)
    conexao.execute("""
        CREATE TABLE IF NOT EXISTS analises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            data_hora TEXT NOT NULL,
            usuario_id TEXT NOT NULL DEFAULT "",
            renda_mensal REAL NOT NULL,
            nivel_endividamento REAL NOT NULL,
            frequencia_poupanca TEXT NOT NULL,
            transacoes_json TEXT NOT NULL,
            perfil_financeiro TEXT NOT NULL,
            probabilidade REAL NOT NULL,
            resumo_gastos_json TEXT NOT NULL,
            recomendacoes_json TEXT NOT NULL,
            alertas_json TEXT NOT NULL DEFAULT "[]"
        )
    """)
    for coluna, definicao in [("alertas_json", "TEXT NOT NULL DEFAULT \\'[]\\'"), ("usuario_id", "TEXT NOT NULL DEFAULT \\'\\'")]:
        try:
            conexao.execute(f"ALTER TABLE analises ADD COLUMN {coluna} {definicao}")
        except sqlite3.OperationalError:
            pass
    conexao.commit()
    conexao.close()


def salvar_analise_no_historico(caminho_db, dados_entrada, resultado, usuario_id=""):
    conexao = sqlite3.connect(caminho_db)
    cursor = conexao.execute(
        """INSERT INTO analises
           (data_hora, usuario_id, renda_mensal, nivel_endividamento, frequencia_poupanca,
            transacoes_json, perfil_financeiro, probabilidade, resumo_gastos_json,
            recomendacoes_json, alertas_json)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(), usuario_id,
            dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"], dados_entrada["frequencia_poupanca"],
            json.dumps(dados_entrada["transacoes"], ensure_ascii=False),
            resultado["perfil_financeiro"], resultado["probabilidade"],
            json.dumps(resultado["resumo_gastos"], ensure_ascii=False),
            json.dumps(resultado["recomendacoes"], ensure_ascii=False),
            json.dumps(resultado["alertas"], ensure_ascii=False),
        )
    )
    conexao.commit()
    id_gerado = cursor.lastrowid
    conexao.close()
    return id_gerado


def listar_historico(caminho_db, limite=20):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linhas = conexao.execute("SELECT * FROM analises ORDER BY id DESC LIMIT ?", (limite,)).fetchall()
    conexao.close()
    return [_linha_para_dict(linha) for linha in linhas]


def obter_analise_por_id(caminho_db, id_analise):
    conexao = sqlite3.connect(caminho_db)
    conexao.row_factory = sqlite3.Row
    linha = conexao.execute("SELECT * FROM analises WHERE id = ?", (id_analise,)).fetchone()
    conexao.close()
    return _linha_para_dict(linha) if linha else None


def _linha_para_dict(linha):
    chaves = linha.keys()
    return {
        "id": linha["id"], "data_hora": linha["data_hora"],
        "usuario_id": linha["usuario_id"] if "usuario_id" in chaves else "",
        "renda_mensal": linha["renda_mensal"], "nivel_endividamento": linha["nivel_endividamento"],
        "frequencia_poupanca": linha["frequencia_poupanca"],
        "transacoes": json.loads(linha["transacoes_json"]),
        "perfil_financeiro": linha["perfil_financeiro"], "probabilidade": linha["probabilidade"],
        "resumo_gastos": json.loads(linha["resumo_gastos_json"]),
        "recomendacoes": json.loads(linha["recomendacoes_json"]),
        "alertas": json.loads(linha["alertas_json"]) if "alertas_json" in chaves and linha["alertas_json"] else [],
    }


def processar_csv_em_lote(conteudo_csv: bytes, artefatos: dict) -> list:
    """Processa um CSV com transacoes de multiplos usuarios, agrupadas por usuario_id.

    Cada usuario_id vira uma analise financeira completa independente,
    reaproveitando a mesma funcao analisar_financas ja validada.
    """
    df = pd.read_csv(io.BytesIO(conteudo_csv))

    colunas_faltando = [c for c in COLUNAS_CSV_OBRIGATORIAS if c not in df.columns]
    if colunas_faltando:
        raise ValueError(f"Colunas obrigatorias ausentes no CSV: {colunas_faltando}")

    resultados = []
    for usuario_id, grupo in df.groupby("usuario_id"):
        primeira_linha = grupo.iloc[0]
        dados_entrada = {
            "renda_mensal": float(primeira_linha["renda_mensal"]),
            "nivel_endividamento": float(primeira_linha["nivel_endividamento"]),
            "frequencia_poupanca": str(primeira_linha["frequencia_poupanca"]),
            "transacoes": [
                {"descricao": str(linha["descricao"]), "valor": float(linha["valor"])}
                for _, linha in grupo.iterrows()
            ],
        }
        resultado = analisar_financas(dados_entrada, artefatos)
        id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_entrada, resultado, usuario_id=str(usuario_id))
        resultados.append({"usuario_id": str(usuario_id), "id_historico": id_historico, **resultado})

    return resultados


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    inicializar_banco_de_dados(CAMINHO_BANCO_DADOS)
    print("Modelos carregados e banco de historico pronto")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    dados_dict = dados.model_dump()
    resultado = analisar_financas(dados_dict, artefatos_globais)
    id_historico = salvar_analise_no_historico(CAMINHO_BANCO_DADOS, dados_dict, resultado)
    return {**resultado, "id_historico": id_historico}


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}


@app.get("/explicabilidade/perfil")
def explicabilidade_perfil():
    return {"modelo": "Random Forest - Perfil Financeiro", "importancia_variaveis": obter_importancia_variaveis_perfil(artefatos_globais)}


@app.get("/explicabilidade/categoria")
def explicabilidade_categoria(top_n: int = Query(default=8, ge=1, le=20)):
    return {"modelo": "Regressao Logistica (TF-IDF) - Categoria de Transacao", "palavras_por_categoria": obter_palavras_influentes_categoria(artefatos_globais, top_n)}


@app.get("/historico")
def historico(limite: int = Query(default=20, ge=1, le=100)):
    return {"analises": listar_historico(CAMINHO_BANCO_DADOS, limite)}


@app.get("/historico/{id_analise}")
def historico_por_id(id_analise: int):
    analise = obter_analise_por_id(CAMINHO_BANCO_DADOS, id_analise)
    if analise is None:
        raise HTTPException(status_code=404, detail=f"Analise com id {id_analise} nao encontrada")
    return analise


@app.post("/analise-financeira/lote")
async def analise_financeira_lote(arquivo: UploadFile = File(...)):
    """Processa um CSV com transacoes de multiplos usuarios em lote.

    Formato esperado do CSV (colunas obrigatorias):
    usuario_id, renda_mensal, nivel_endividamento, frequencia_poupanca, descricao, valor

    Cada usuario_id agrupa suas proprias transacoes e recebe uma
    analise financeira completa e independente, salva no historico.
    """
    if not arquivo.filename.endswith(".csv"):
        raise HTTPException(status_code=400, detail="O arquivo deve ser um CSV (.csv)")

    conteudo = await arquivo.read()
    try:
        resultados = processar_csv_em_lote(conteudo, artefatos_globais)
    except ValueError as erro:
        raise HTTPException(status_code=422, detail=str(erro))
    except Exception as erro:
        raise HTTPException(status_code=422, detail=f"Erro ao processar o CSV: {str(erro)}")

    return {"total_processado": len(resultados), "resultados": resultados}
'''


CONTEUDO_TESTES = '''
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


def test_status_no_ar(cliente):
    assert cliente.get("/").status_code == 200


def test_lote_csv_processa_multiplos_usuarios(cliente):
    csv_conteudo = (
        "usuario_id,renda_mensal,nivel_endividamento,frequencia_poupanca,descricao,valor\\n"
        "usuario_1,4500,25,Media,Supermercado,420\\n"
        "usuario_1,4500,25,Media,Combustivel,300\\n"
        "usuario_2,3000,62,Baixa,Aluguel Residencial,1200\\n"
    )
    arquivos = {"arquivo": ("lote_teste.csv", csv_conteudo, "text/csv")}
    resposta = cliente.post("/analise-financeira/lote", files=arquivos)
    assert resposta.status_code == 200
    corpo = resposta.json()
    assert corpo["total_processado"] == 2
    usuarios_processados = {r["usuario_id"] for r in corpo["resultados"]}
    assert usuarios_processados == {"usuario_1", "usuario_2"}


def test_lote_csv_gera_id_historico_por_usuario(cliente):
    csv_conteudo = (
        "usuario_id,renda_mensal,nivel_endividamento,frequencia_poupanca,descricao,valor\\n"
        "usuario_A,5000,10,Alta,Streaming,40\\n"
    )
    arquivos = {"arquivo": ("lote2.csv", csv_conteudo, "text/csv")}
    resposta = cliente.post("/analise-financeira/lote", files=arquivos)
    corpo = resposta.json()
    assert "id_historico" in corpo["resultados"][0]


def test_lote_csv_rejeita_arquivo_nao_csv(cliente):
    arquivos = {"arquivo": ("dados.txt", "conteudo qualquer", "text/plain")}
    resposta = cliente.post("/analise-financeira/lote", files=arquivos)
    assert resposta.status_code == 400


def test_lote_csv_rejeita_colunas_faltando(cliente):
    csv_conteudo = "usuario_id,descricao,valor\\nusuario_1,Netflix,40\\n"
    arquivos = {"arquivo": ("incompleto.csv", csv_conteudo, "text/csv")}
    resposta = cliente.post("/analise-financeira/lote", files=arquivos)
    assert resposta.status_code == 422
'''


def escrever_arquivos() -> None:
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ Arquivos escritos localmente")


def executar_suite_de_testes() -> bool:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    return resultado.returncode == 0


def enviar_e_reiniciar(ip_servidor: str, usuario: str, caminho_chave: str) -> None:
    chave = paramiko.RSAKey.from_private_key_file(caminho_chave)
    cliente_ssh = paramiko.SSHClient()
    cliente_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cliente_ssh.connect(hostname=ip_servidor, username=usuario, pkey=chave, timeout=15)

    sftp = cliente_ssh.open_sftp()
    sftp.put("/content/main.py", "/home/ubuntu/main.py")
    sftp.close()
    print("✅ main.py enviado para o servidor")

    cliente_ssh.exec_command("pip install python-multipart --quiet 2>&1 | tail -1")
    time.sleep(1)

    cliente_ssh.exec_command("pkill -f uvicorn")
    time.sleep(2)

    comando_subir = (
        "cd /home/ubuntu && source venv/bin/activate && pip install python-multipart --quiet && "
        "nohup uvicorn main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 < /dev/null &"
    )
    cliente_ssh.exec_command(comando_subir)
    time.sleep(3)
    cliente_ssh.close()
    print("✅ API reiniciada no servidor (com python-multipart instalado)")


def confirmar_producao(ip_servidor: str) -> None:
    time.sleep(6)
    try:
        resposta_saude = requests.get(f"http://{ip_servidor}:8000/", timeout=10)
        print("🔍 Health check:", resposta_saude.json())

        csv_teste = (
            "usuario_id,renda_mensal,nivel_endividamento,frequencia_poupanca,descricao,valor\n"
            "usuario_1,4500,25,Media,Supermercado,420\n"
            "usuario_2,3000,62,Baixa,Aluguel Residencial,1200\n"
        )
        arquivos = {"arquivo": ("teste_lote.csv", csv_teste, "text/csv")}
        resposta_lote = requests.post(f"http://{ip_servidor}:8000/analise-financeira/lote", files=arquivos, timeout=15)
        corpo = resposta_lote.json()
        print("\n🔍 Teste de lote via CSV:")
        print(corpo)

        if resposta_lote.status_code == 200 and corpo.get("total_processado") == 2:
            print("\n🎉 Item 5 (Processamento em Lote via CSV) CONFIRMADO em produção!")
        else:
            print(f"\n⚠️ Resultado inesperado — status {resposta_lote.status_code}")

    except Exception as e:
        print(f"❌ Erro ao testar produção: {e}")


# ------------------------------------------------------------
# EXECUÇÃO COMPLETA
# ------------------------------------------------------------
IP_SERVIDOR = "140.238.178.157"
USUARIO = "ubuntu"
CAMINHO_CHAVE = "ssh-key-2026-08-01.key"

instalar_dependencias()
escrever_arquivos()
testes_ok = executar_suite_de_testes()

if testes_ok and os.path.exists(CAMINHO_CHAVE):
    print("\n" + "=" * 60)
    print("🚀 Testes locais OK — iniciando deploy para produção")
    print("=" * 60)
    enviar_e_reiniciar(IP_SERVIDOR, USUARIO, CAMINHO_CHAVE)
    confirmar_producao(IP_SERVIDOR)
elif not testes_ok:
    print("\n❌ Testes locais falharam — deploy NÃO realizado, corrija antes de continuar")
else:
    print(f"\n⚠️ Chave '{CAMINHO_CHAVE}' não encontrada — faça upload antes: from google.colab import files; files.upload()")

ModuleNotFoundError: No module named 'paramiko'